# Day 053 — Exercise 3: A Uniform Response Envelope

**What you'll build:** `request_json(client, method, path, payload)` — a single helper that calls any backend route and normalises *every* outcome into one shape: `{'ok', 'status', 'data', 'error'}`.

**Why it matters:** A frontend calls many endpoints, and each can succeed, return a 4xx/5xx, or fail to connect. Handling those three cases inline at every call site is a mess. One envelope function means the UI branches on `if env['ok']:` everywhere — the same discipline as the `(is_valid, result)` tuple from Day 51, generalised to HTTP.

## Provided: Setup + Backend + check_health + post_chat

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import httpx
import ollama


# ---- The AI backend (built on Day 52 — provided here) ----
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    reply: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app


def check_health(client) -> bool:
    """Ping the backend's GET /health through an injected HTTP client.

    Returns True only if the request succeeds with 200 AND status == 'ok'.
    Any exception (backend down, connection refused) -> False, never raises.
    The `client` is duck-typed: an httpx.Client in production, a TestClient in
    tests — both expose .get / .post / .request.
    """
    try:
        resp = client.get('/health')
        return resp.status_code == 200 and resp.json().get('status') == 'ok'
    except Exception:
        return False


def post_chat(client, message: str, temperature: float = 0.7) -> dict:
    """POST /chat honouring the JSON contract {message, temperature}.

    Returns the parsed {reply, model} on 200. On a non-200 status returns
    {'error': ..., 'status': code}; on a connection failure returns
    {'error': ...}. The frontend never sees a raw exception.
    """
    try:
        resp = client.post('/chat', json={'message': message, 'temperature': temperature})
    except Exception as e:
        return {'error': f'request failed: {e}'}
    if resp.status_code != 200:
        return {'error': f'backend returned {resp.status_code}', 'status': resp.status_code}
    return resp.json()

## Your Implementation

In [ ]:
def request_json(client, method: str, path: str, payload: dict = None) -> dict:
    """
    Normalise every outcome into:
        {'ok': bool, 'status': int|None, 'data': dict|None, 'error': str|None}
    - 2xx           -> ok=True,  status=code, data=json
    - 4xx/5xx       -> ok=False, status=code, error='HTTP <code>'
    - conn failure  -> ok=False, status=None, error='connection error: ...'
    """
    # TODO: try:
    #     resp = client.request(method, path, json=payload)
    # TODO: except Exception as e:
    #     return {'ok': False, 'status': None, 'data': None, 'error': f'connection error: {e}'}
    # TODO: ok = 200 <= resp.status_code < 300
    # TODO: try: data = resp.json()
    #       except Exception: data = None
    # TODO: return {'ok': ok, 'status': resp.status_code,
    #               'data': data if ok else None,
    #               'error': None if ok else f'HTTP {resp.status_code}'}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    backend = TestClient(build_api())

    # Check 1: success envelope for GET /templates
    try:
        env = request_json(backend, 'GET', '/templates')
        assert env['ok'] is True and env['status'] == 200, f'bad envelope: {env}'
        assert env['data'] is not None and env['error'] is None
        passed += 1; print('✅ Check 1: 2xx -> ok=True with data')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: envelope always has the four keys
    try:
        env = request_json(backend, 'GET', '/templates')
        for k in ('ok', 'status', 'data', 'error'):
            assert k in env, f'missing key: {k}'
        passed += 1; print('✅ Check 2: envelope has ok/status/data/error')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: error status -> ok=False with status + error
    try:
        env = request_json(backend, 'GET', '/no-such-route')
        assert env['ok'] is False and env['status'] == 404, f'expected 404 envelope: {env}'
        assert env['data'] is None and env['error'] is not None
        passed += 1; print('✅ Check 3: 404 -> ok=False, status=404')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: connection failure -> ok=False, status=None
    try:
        class _Dead:
            def request(self, *a, **k):
                raise httpx.ConnectError('refused')
        env = request_json(_Dead(), 'GET', '/health')
        assert env['ok'] is False and env['status'] is None, f'bad conn envelope: {env}'
        passed += 1; print('✅ Check 4: connection failure -> ok=False, status=None')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: POST with a payload works (round-trips through /chat)
    try:
        env = request_json(backend, 'POST', '/chat', {'message': 'Say hi in 3 words.'})
        assert env['ok'] is True and env['status'] == 200, f'expected 200: {env}'
        assert 'reply' in env['data']
        passed += 1; print('✅ Check 5: POST with payload -> ok=True')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def request_json(client, method: str, path: str, payload: dict = None) -> dict:
    """Call the backend and normalise EVERY outcome into one envelope:

        {'ok': bool, 'status': int | None, 'data': dict | None, 'error': str | None}

    - success (2xx):        ok=True,  status=code, data=json
    - error status (4xx/5xx): ok=False, status=code, error='HTTP <code>'
    - connection failure:   ok=False, status=None, error='connection error: ...'

    One shape for the whole frontend to branch on — no scattered try/except.
    """
    try:
        resp = client.request(method, path, json=payload)
    except Exception as e:
        return {'ok': False, 'status': None, 'data': None,
                'error': f'connection error: {e}'}
    ok = 200 <= resp.status_code < 300
    try:
        data = resp.json()
    except Exception:
        data = None
    return {
        'ok':     ok,
        'status': resp.status_code,
        'data':   data if ok else None,
        'error':  None if ok else f'HTTP {resp.status_code}',
    }
```

**Why this works:** `client.request(method, path, json=payload)` is the generic form of `.get`/`.post`, so one function covers every verb. The envelope collapses three failure modes into a single predictable shape: the caller checks `env['ok']` and reads `env['data']` or `env['error']` — never a try/except at the call site. This is the backbone the `AIAppClient` class is built on in Exercise 5.
</details>